# JDL25 Lab. 3: Sequence modeling with Recurrent Neural Networks and Transformers

In [ ]:
from __future__ import print_function
import tensorflow as tf
import os, json, re
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from keras.preprocessing.text import Tokenizer # If you work on colab you might need to use from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import pandas as pd

## 1. Deal with sequential data

<img src="https://miro.medium.com/v2/resize:fit:1222/0*amLuu-tSiYZyny8W.png" width="600px" align="right"><br>
In this lab we see Deep Learning models that can process sequential data (text, timeseries,..).<br>
These models don’t take as input raw text: they only work with numeric tensors; **vectorizing** text is the process of transforming text into numeric tensors.<br><br><br>
The different units into which you can break down text (words, characters) are called tokens; then if you apply a tokenization scheme, you associate numeric vectors with the generated tokens.<br>
These vectors, packed into sequence tensors, are fed into Deep Neural Network.<br>
There are multiple ways to associate a vector with a token, we will use an approach based on Word Embedding.<br>

###  1.1 Word embedding
<img src="https://jiajingchen.github.io/assets/images/post_image/one-hot-vs-word-embeddings.png" width="400px" align="right"><br>
The vector obtained from word embedding is dense.  There are two ways to obtain word embeddings:<br>
* They may be learned jointly with the network
* We may use pre-trained word vectors (Word2Vec, GloVe,..)


Word embeddings maps human language into a geometric space; in a reasonable embedding space synonyms are embedded into similar word vectors and the geometric distance between any two word vectors reflects the semantic distance between the associated words (words meaning different things are embedded at points far away from each other, whereas related words are closer).<br>
As usual, how good is a word-embedding space depends on the specific task.<br>

### 1.2 Learning Word Embeddings with the embedding layer
#### Load imdb dataset
This dataset contains movies reviews from IMDB, labeled by sentiment(positive/negative); reviews have been preprocessed, and each review is encoded as a sequence of word indexes (integers).<br>
https://keras.io/datasets/#imdb-movie-reviews-sentiment-classification

In [ ]:
max_features = 10000
maxlen = 50

imdb = tf.keras.datasets.imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

#### Show the size of vocabulary and the most frequent words

In [ ]:
word_to_index = imdb.get_word_index()

vocab_size = len(word_to_index)
print('Vocab size : ', vocab_size)


words_freq_list =words_freq_list = []
[]
for (k,v) in imdb.get_word_index().items():
    words_freq_list.append((k,v))

sorted_list = sorted(words_freq_list, key=lambda x: x[1])

print("50 most common words: \n")
print(sorted_list[0:50])

## 2. Recurrent Neural Network
Here https://colah.github.io/posts/2015-08-Understanding-LSTMs/ you can find a clear explanation about RNNs and LSTMs; the following is a summary of the main concepts. <br>

A recurrent neural network (RNN) processes sequences by iterating through the sequence elements and maintaining a state containing information related to what it has seen so far.

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTSEBQgTGbZhPZjxr8C7h7UFntX28R0LWTnvzSIIfOE&s" width="650px"><br>


Each input is combined with the internal state and then is applied an activation function (e.g. $tanh$); then the output is computed and the internal state is updated.<br>
In many cases, you just need the last output ($h_{i=last t}$ at the end of the loop), because it already contains information
about the entire sequence.
<img src="https://user-images.githubusercontent.com/15166794/39033685-30859e24-44ae-11e8-9d7d-860c75efe080.png" width="550px">


### 2.1 Create the model
In the following sections we will develop different models. Be careful to the fact that we are dealing with a binary classification problem!

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Embedding(max_features, 16))

# Complete the model, it should be made by at least:
# 1 SimpleRNN layer --> Check the keras documentation for the syntax
# 1 Dense layer --> Reason on the most appropriate activation function 

model.add(...)
model.add(...)

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])

### 2.2 Train the model

In [ ]:
# Train your model here

history = model.fit(x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2)


In [ ]:
def plot_history(history):
    # Plot training & validation accuracy values
    plt.plot(history.history['acc'])
    plt.plot(history.history['val_acc'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Val'], loc='upper left')
    plt.show()

    # Plot training & validation loss values
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Val'], loc='upper left')
    plt.show()

In [ ]:
plot_history(history)

### 2.3 Evaluate the model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)
print('Test accuracy: %.3f, Test loss: %.3f' % (test_acc,test_loss))

### 2.4 Try to build a new model where you stack several recurrent layers.
In such a setup, you have to get all of the intermediate layers to return full sequence of outputs. This is needed to return batch size, timesteps, hidden state. By doing this the output should contain all historical generated outputs along with time stamps. This way the next layer can work further on the data.<br>

If you have time, you might want to try with LSTMs cells.

In [ ]:
# Build the model. It should be made by at least:
# 1 Embedding layer
# More than 1 SimpleRNN layer, do not forget to put the return_sequences parameter to True in all 
# recurrent layers but the last one
# 1 Dense layer

model = tf.keras.Sequential()
model.add(tf.keras.layers.Embedding(max_features, 16))

# ... add code here ...

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])

#### Train the model

In [ ]:
# Train your model here

history = model.fit(x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2)




In [ ]:
plot_history(history)

#### Evaluate the model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)
print('Test accuracy: %.3f, Test loss: %.3f' % (test_acc,test_loss))

**What can you say about the obtained results? What about the comparison between these results and the ones obtained in the single layer RNN?**

## 3. Reuters newswire classification dataset

The reuters newswire classification dataset is a dataset of 11,228 newswires from Reuters, labeled over 46 topics. More information about the dataset and how to use it can be found here:
https://keras.io/api/datasets/reuters/

Try to build a new model dealing with this new dataset.
Be carefull that this will imply some changes in your code as it is not a binary classification problem anymore!

In [ ]:
imdb = tf.keras.datasets.imdb
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.reuters.load_data(num_words=max_features)

num_classes = np.max(y_train) + 1
print(num_classes)

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

#### Create model

In [ ]:
# Build the model.


#### Train the model

In [ ]:
# Train your model here


In [ ]:
plot_history(history)

#### Evaluate the model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)
print('Test accuracy: %.3f, Test loss: %.3f' % (test_acc,test_loss))

**Comment the results!**

## 4. Using Transformers

<img src="https://machinelearningmastery.com/wp-content/uploads/2021/08/attention_research_1-768x1082.png" width="250px" align="right"><br>

One main drawback about RNNs is their capacity to remember long-term dependencies. To alleviate this problem different models have been proposed, like Long Short Term Memories (LSTM) and Transformers.<br>
*Transformers* is one of the best available model nowadays to deal with different kind of data (text, images..) and obtain state of the art results.

The key component in the Transformer architecture is the Attention layer, that helps the encoder look at other words in the input sentence as it encodes a specific word. The attention mechanism, in theory, and given enough compute resources, have a wider window to reference from, therefore being capable of using the **entire context** of the text.

In [ ]:
# Define the Transformer block in TensorFlow

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [tf.keras.layers.Dense(ff_dim, activation="relu"), tf.keras.layers.Dense(embed_dim),]
        )
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
# Define the input embeddings and the positional encoding

class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [ ]:
# Define an architecture with the representation part employing the Transformer, followed by a classifier

embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer

# Build the model. It should be made by at least:
# 1 TokenAndPositionEmbedding layer with maxlen, vocab_size and embed_dim as arguments <-- ALREADY DONE
# 1 TransformerBlock layer with embed_dim, num_heads and ff_dim as arguments <-- ALREADY DONE
# 1 GlobalAveragePooling1D layer
# 2 Dense layers (including the last one)

model = tf.keras.Sequential()
model.add(TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim))
model.add(TransformerBlock(embed_dim, num_heads, ff_dim))
model.add(...)
model.add(...)
model.add(...)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])

# If needed, add Dropout for regularization